# AI Product Manager · 从 Idea 到 PRD

## 第1部分：项目介绍

作者：[dandan1232](https://github.com/dandan1232) · 日期：2026-11-23

本 Notebook 调用与网页相同的真实工作流：逐步访谈、用户确认、可选联网研究、可行性分析、PRD 与一致性复核。

适用于希望把模糊想法整理成明确需求的个人开发者与学习者，不需要准备知识库。输入你自己的真实想法；没有密钥时会要求输入，不展示伪造结果。逐格运行，提交前清空输出。

## 第2部分：环境配置

在项目目录、同一个 Python 环境中运行 `python -m pip install -r requirements.txt`，然后执行 `python -m jupyter lab main.ipynb`。清单包含 HelloAgents、JupyterLab 和 IPython；不需要安装示例里的全部扩展、绘图库或 Web 框架。

以下导入复用 src 中的实现；SQLite、HTTP 搜索和配置文件读取使用标准库。

In [ ]:
from pathlib import Path
import sys

candidates = [Path.cwd(), Path.cwd() / 'Co-creation-projects' / 'dandan1232-AIProductManager']
ROOT = next((p for p in candidates if (p / 'src' / 'workflow.py').exists()), None)
if ROOT is None:
    raise RuntimeError('请从本项目目录或仓库根目录打开 Notebook。')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import Settings
from src.services import HelloAgentsBackend, TavilySearch
from src.workflow import Store, Workflow
from src.prompts import BRIEF_KEYS, BRIEF_LABELS


### 2.1 连接模型

可在项目 `.env` 设置模型，也可通过下方隐藏输入填写密钥。密钥只保存在当前内核中，不输出、不写入项目记录。

In [ ]:
from getpass import getpass

settings = Settings.from_env()
if not settings.model:
    settings.model = input('模型名称：').strip()
if not settings.api_key:
    settings.base_url = input('OpenAI 兼容接口基础地址（留空使用当前默认值）：').strip() or settings.base_url
    settings.api_key = getpass('模型 API Key：').strip()
settings.validate()
print('配置已读取，尚未发送模型请求。')


## 第3部分：工具定义

搜索工具定义在 `src/services.py` 的 `TavilySearch` 类中。`search(query)` 调用真实 Tavily API，返回来源编号、标题、URL、摘录和查询时间，过滤无效 URL 与重复来源。没有密钥会明确报错。

本项目由工作流明确调用工具，不依赖模型自主选择工具，因此使用普通 Python 适配类，不需要照搬模板的 BaseTool 空壳。下面只创建实例；是否联网由功能演示时的选择决定。

In [ ]:
search_tool = TavilySearch(settings.search_key)


## 第4部分：智能体构建

采用固定阶段的多智能体工作流：需求澄清 → 用户确认 → 竞品研究 → 可行性评估 → PRD 编写 → 一致性复核。

`HelloAgentsBackend.run(role, context)` 为每次调用创建真实的 HelloAgentsLLM 和 SimpleAgent。五种角色的提示词在 `src/prompts.py`；工作流在 `src/workflow.py` 中按顺序传递确认需求和研究结果。每次调用独立创建智能体，避免项目历史混用。

下面组装模型适配器、搜索工具与本地存储，不立即发送模型请求。具体类定义保留在 src，Notebook 不复制一份实现。

In [ ]:
backend = HelloAgentsBackend(settings)
flow = Workflow(Store(ROOT / 'runtime' / 'notebook.sqlite3'), backend, search_tool)
print('工作流已构建。下一格会实际调用模型服务。')


## 第5部分：功能演示

### 5.1 输入 Idea 并逐步回答

AI 每轮只问一个关键问题，最多六轮。输入空回答可提前结束访谈，进入需求检查。发生模型错误时保留当前 state，修正连接后可以调用 `flow.interview(state)` 重试。

In [ ]:
idea = input('描述你想做的产品：').strip()
state = flow.create(idea)
flow.interview(state)
while True:
    print('\n' + state['messages'][-1]['content'])
    if state['ready']:
        break
    for index, option in enumerate(state['options'], 1):
        print(f'{index}. {option}')
    answer = input('你的回答（直接回车结束访谈）：').strip()
    if not answer:
        break
    flow.add_answer(state, answer)
    flow.interview(state)


### 5.2 检查并确认需求

下面逐项展示草稿。直接回车保留内容；填写新内容进行修改。至少填写产品名称和要解决的问题。只有明确输入“确认”才进入后续分析。

In [ ]:
confirmed = dict(state['brief'])
for key, label in zip(BRIEF_KEYS, BRIEF_LABELS):
    print(f'\n{label}：{confirmed[key] or "尚未确定"}')
    replacement = input('修改为（回车保留）：').strip()
    if replacement:
        confirmed[key] = replacement
if input('\n确认以这份需求作为分析依据？请输入“确认”：').strip() != '确认':
    raise RuntimeError('需求尚未确认。请修改后重新运行此格。')
flow.confirm(state, confirmed)
print('需求快照已保存。')


### 5.3 研究、评估并生成 PRD

联网研究需要 Tavily 密钥；不联网时，文档会明确写出未开展竞品研究。研究和模型调用可能产生服务商费用。分析中的新增细节为待评审建议。

In [ ]:
online = input('是否联网查询竞品？输入 y 开启，其余跳过：').strip().lower() == 'y'
if online and not settings.search_key:
    settings.search_key = getpass('Tavily API Key：').strip()
    if not settings.search_key:
        raise RuntimeError('联网研究需要搜索密钥，请重新选择。')
flow.search = TavilySearch(settings.search_key)
flow.generate(state, online=online)
from IPython.display import Markdown, display
display(Markdown(state['report']))


### 5.4 导出真实生成结果

运行结果保存在被 Git 忽略的 `outputs/`。需要分享时先检查内容，避免连同个人访谈或敏感需求一起公开。

In [ ]:
output_dir = ROOT / 'outputs'
output_dir.mkdir(exist_ok=True)
output_path = output_dir / f"prd-{state['id']}.md"
output_path.write_text(state['report'], encoding='utf-8')
print(f'已保存：{output_path}')
if state['conflicts']:
    print('一致性检查发现冲突，请查看报告并修改需求或重新生成。')


## 第6部分：验证、性能评估与限制

自动化测试：`python -m unittest discover -s tests -t . -v`。

测试替身只验证流程行为，不证明模型回答质量。真实竞品依赖搜索结果质量；引用编号存在不代表完成了自动事实核验。商业需求需要真实用户验证。尚无真实模型端到端运行记录，不提供未经测量的准确率和响应时间。

架构与验收详见 `docs/architecture.md`、`docs/requirements.md` 和 `docs/verification.md`。

## 第7部分：总结与展望

已实现：逐步访谈、确认快照、可选竞品研究、可行性建议、PRD、一致性复核和 Markdown 导出；网页和 Notebook 共用工作流。

实现中的挑战与处理：模型建议可能偏离用户决定，因此确认内容独立保存并直接写入报告；搜索可能失败，因此明确标记缺失资料；模型响应可能不符合结构，因此检查关键字段并保留可重试状态。以上措施不保证语义质量。

下一步：配置真实服务完成全流程验证，保存去除敏感信息的实际输出与耗时；随后评估需求版本对比、分阶段继续生成与更多搜索工具。